## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

## Load and Basic Cleaning

In [ ]:
df = pd.read_csv('beijing_dataset.csv')
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
df.drop(columns='No', inplace=True)

In [ ]:
df = df.sort_values(['year', 'month', 'day', 'hour']).reset_index(drop=True)


In [ ]:
df.shape

In [ ]:
print(f"Year range {df['year'].min()} - {df['year'].max()} ")
print(f"Month range {df['month'].min()} - {df['month'].max()}")
print(f"Day range {df['day'].min()} - {df['day'].max()} ")
print(f"Hour range {df['hour'].min()} - {df['hour'].max()} ")

## Exploratory Data Analysis

In [ ]:
df.isnull().sum()

In [ ]:
df['pm2.5'] = df['pm2.5'].interpolate(method='linear', limit_direction='both')

### 3.1 PM2.5 Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['pm2.5'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('PM2.5', fontsize=18)
axes[0].set_xlabel('PM2.5 (ug/m3)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['pm2.5'].mean(), color='red', linestyle='--', label=f"Mean: {df['pm2.5'].mean():.1f}")
axes[0].legend()

axes[1].hist(np.log1p(df['pm2.5']), bins=60)
axes[1].set_title('log(1 + PM2.5)', fontsize=14)
axes[1].set_xlabel('log(1 + PM2.5)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"Skewness (raw): {df['pm2.5']}")

### 3.2 Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
corr = df.select_dtypes(include=['int64', 'float64']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr,mask=mask ,annot=True, fmt='.2f')
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

print('Top correlations with PM2.5')
print(corr['pm2.5'].drop('pm2.5').abs().sort_values(ascending=False).to_string())


### 3.3 Temporal Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 16))

# monthly pattern 
df.groupby('month')['pm2.5'].mean().plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Average PM2.5 by Month')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('PM2.5')

# Hourly Pattern
df.groupby('hour')['pm2.5'].mean().plot(kind='bar', ax=axes[0,1], color='darkorange')
axes[0,1].set_title('Average PM2.5 by Hour of Day')
axes[0,1].set_xlabel('Hour')
axes[0,1].set_ylabel('PM2.5 (µg/m³)')

# Year-over-year
df.groupby('year')['pm2.5'].mean().plot(kind='bar', ax=axes[1,0], color='seagreen')
axes[1,0].set_title('Average PM2.5 by Year')
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('PM2.5 (µg/m³)')

# Wind direction
sns.boxplot(x=df['cbwd'], y=df['pm2.5'], ax=axes[1, 1], order=['NE', 'NW', 'SE', 'cv'])
axes[1, 1].set_title('PM2.5 by wind direction')
axes[1, 1].set_xlabel('Wind Direction')
axes[1, 1].set_ylabel('PM2.5')
axes[1, 1].set_ylim(0, 400)

plt.tight_layout()
plt.show()


### 3.4 Meteorological Feature Relationship

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['TEMP', 'DEWP', 'PRES']):
    ax.scatter(df[col], df['pm2.5'], alpha=0.1, s=5, color='steelblue')
    ax.set_xlabel(col)
    ax.set_ylabel('PM2.5')
    ax.set_title(f'{col} vs PM2.5')
    ax.set_ylim(0, 600)

plt.suptitle("Meteorological FEatures vs PM2.5", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(df['Iws'], df['pm2.5'], alpha=0.1, s=5)
plt.xlabel('Cummulative Wind Speed (Iws)')
plt.ylabel('PM2.5')
plt.title('Wind Speed vs PM2.5')
plt.ylim(0, 600)
plt.xlim(0, 100)
plt.show()

### 3.5 Temporal Autocorrelation

In [ ]:
lags = range(1, 49)
autocorr = [df['pm2.5'].autocorr(lag=lag) for lag in lags]

plt.figure(figsize=(12, 4))
plt.bar(lags, autocorr, color='steelblue')
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel('Lag (hours)')
plt.ylabel('Autocorrelation')
plt.title('PM2.5 Autocorrelation')
plt.xticks(list(range(1, 49, 1)))
plt.tight_layout()
plt.show()

print(f"Autocorrelation at lag-1: {df['pm2.5'].autocorr(1):.3f}")
print(f"Autocorrelation at lag-2: {df['pm2.5'].autocorr(2):.3f}")
print(f"Autocorrelation at lag-24: {df['pm2.5'].autocorr(24):.3f}")

### 4. Feature Engineering

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df.drop(columns=['hour', 'month'], inplace=True)

for lag in [1, 2, 3, 6, 12, 24]:
    df[f'pm25_lag{lag}'] = df['pm2.5'].shift(lag)

df['pm25_roll3'] = df['pm2.5'].shift(1).rolling(window=3).mean()
df['pm25_roll6'] = df['pm2.5'].shift(1).rolling(window=6).mean()
df['pm25_roll12'] = df['pm2.5'].shift(1).rolling(window=12).mean()
df['pm25_roll24'] = df['pm2.5'].shift(1).rolling(window=24).mean()

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Dataset shape after feature engineering: {df.shape}")
print(f"New features: {[c for c in df.columns if 'lag' in c or 'roll' in c or 'sin' in c or 'cos' in c]}")

### 5. Temporal Train/Test split + log-transform target

In [ ]:
split_idx = int(len(df)*0.8)
df_train = df.iloc[:split_idx]
df_test = df.iloc[split_idx:]

print(f"Train: {df_train['year'].min()} - {df_train['year'].max()} ({len(df_train)}) rows")
print(f'Test:  {df_test["year"].min()}–{df_test["year"].max()} ({len(df_test)} rows)')

X_train = df_train.drop('pm2.5', axis=1)
X_test = df_test.drop('pm2.5', axis=1)

y_train = np.log1p(df_train['pm2.5'].values)
y_test = np.log1p(df_test['pm2.5'].values)
y_test_raw = df_test['pm2.5'].values # for final evaluation

### 6. Preprocessing Pipeline

In [ ]:
numerical_cols   = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print('Numerical features:  ', numerical_cols)
print('Categorical features:', categorical_cols)

numerical_pipeline = Pipeline(steps=[('scaler', RobustScaler())])
categorical_pipeline = Pipeline(steps=[('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline,   numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'\nProcessed train shape: {X_train_processed.shape}')
print(f'Processed test shape:  {X_test_processed.shape}')

### 7. Model Architecture

In [ ]:
model = Sequential([
    Input(shape=(X_train_processed.shape[1],)),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.1),

    Dense(1, activation='linear')
])

model.summary()

### 8. Compile and Train

In [ ]:
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    X_train_processed, y_train,
    validation_data=(X_test_processed, y_test),
    epochs=200,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

In [ ]:

y_pred_log = model.predict(X_test_processed).flatten()
y_pred = np.expm1(y_pred_log)   
y_pred = np.maximum(y_pred, 0)   

mae  = mean_absolute_error(y_test_raw, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred))
r2   = r2_score(y_test_raw, y_pred)

print('Test Set Evaluation')
print(f'  MAE  : {mae:.2f}  µg/m³')
print(f'  RMSE : {rmse:.2f} µg/m³')
print(f'  R²   : {r2:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss (MSE in log space)')
axes[0].plot(history.history['val_loss'], label='Val Loss (MSE in log space)')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE (log space)')
axes[0].legend()

axes[1].plot(history.history['mae'],     label='Train MAE (log space)')
axes[1].plot(history.history['val_mae'], label='Val MAE (log space)')
axes[1].set_title('MAE Curve (log space)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test_raw - y_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Predicted vs Actual
axes[0].scatter(y_test_raw, y_pred, alpha=0.2, s=5)
axes[0].plot([0, y_test_raw.max()], [0, y_test_raw.max()], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual PM2.5 (µg/m³)')
axes[0].set_ylabel('Predicted PM2.5 (µg/m³)')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

# Residual plot
axes[1].scatter(y_pred, residuals, alpha=0.2, s=5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted PM2.5 (µg/m³)')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Plot')

# Residual distribution
axes[2].hist(residuals, bins=60, color='steelblue', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('Residual (µg/m³)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Distribution')

plt.suptitle('Residual Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'Mean residual: {residuals.mean():.2f} µg/m³ (should be near 0 = unbiased)')
print(f'Std  residual: {residuals.std():.2f} µg/m³')

In [ ]:
from scipy import stats
plt.figure(figsize=(6,6))
stats.probplot(y_test_raw - y_pred, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()